In [16]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import matplotlib.pyplot as plt #for plots
from sklearn.model_selection import train_test_split # for train test split
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        pass
        #print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [17]:
from pathlib import Path
image_folder_path = Path(r"/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000")
assets_folder=Path(r"/kaggle/input/datasets/ayomidefagbolade/ham10000-outputs/outputs")
test_split = pd.read_csv(assets_folder / "test_split.csv")
train_split = pd.read_csv(assets_folder / "train_split.csv")
val_split   = pd.read_csv(assets_folder / "val_split.csv")
label_mapping= pd.read_csv(assets_folder/"label_mapping.csv")


In [18]:
val_split.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,image_path,label
0,HAM_0007391,ISIC_0031926,nv,follow_up,55.0,female,trunk,/kaggle/input/datasets/kmader/skin-cancer-mnis...,5
1,HAM_0005063,ISIC_0030419,nv,follow_up,60.0,male,upper extremity,/kaggle/input/datasets/kmader/skin-cancer-mnis...,5
2,HAM_0005870,ISIC_0030399,nv,histo,30.0,male,upper extremity,/kaggle/input/datasets/kmader/skin-cancer-mnis...,5
3,HAM_0003077,ISIC_0031230,nv,follow_up,40.0,female,lower extremity,/kaggle/input/datasets/kmader/skin-cancer-mnis...,5
4,HAM_0000349,ISIC_0033005,df,consensus,45.0,female,lower extremity,/kaggle/input/datasets/kmader/skin-cancer-mnis...,3


In [19]:
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

class HAM10000ImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        label = int(row["label"])

        if self.transform:
            image = self.transform(image)

        return image, label
        

CUDA available: True
Device: Tesla T4


In [20]:


from collections import Counter
from tqdm.auto import tqdm

all_paths = pd.concat([
    train_split["image_path"],
    val_split["image_path"],
    test_split["image_path"],
]).tolist()

size_counter = Counter()

for path in tqdm(all_paths):
    with Image.open(path) as img:
        size_counter[img.size] += 1

size_counter

  0%|          | 0/9958 [00:00<?, ?it/s]

Counter({(600, 450): 9958})

Following common  transfer-learning practice, all dermoscopic images were resized from their original 600×450 resolution to 224×224 before model training. Following common  transfer-learning practice, images were resized to 224×224 and normalized using ImageNet mean and standard deviation because the image-only baseline uses an ImageNet-pretrained backbone. 

In [21]:


IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
BATCH_SIZE= 32

train_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [22]:
train_ds = HAM10000ImageDataset(train_split, transform=train_tfms)
val_ds = HAM10000ImageDataset(val_split, transform=eval_tfms)
test_ds = HAM10000ImageDataset(test_split, transform=eval_tfms)

In [23]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

images, labels = next(iter(train_loader))
print(images.shape)
print(labels[:10])

torch.Size([32, 3, 224, 224])
tensor([4, 5, 5, 5, 5, 5, 5, 5, 5, 5])


In [24]:


NUM_CLASSES = int(label_mapping["label"].nunique()) if "label" in label_mapping.columns else train_split["label"].nunique()

class_counts = train_split["label"].value_counts().sort_index()
class_weights = len(train_split) / (NUM_CLASSES * class_counts)

class_weights = torch.tensor(class_weights.values, dtype=torch.float32)
print(class_counts)
print(class_weights)

label
0     261
1     411
2     871
3      92
4     889
5    5328
6     114
Name: count, dtype: int64
tensor([ 4.3602,  2.7689,  1.3065, 12.3696,  1.2801,  0.2136,  9.9825])


In [29]:

import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = train_split["label"].nunique()

weights = models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
model = models.efficientnet_v2_s(weights=weights)
print(model.classifier)          # prints the whole classifier block
print(model.classifier[-1])

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)

model = model.to(device)

print(model.classifier)          # prints the whole classifier block
print(model.classifier[-1])      # prints the last layer inside classifier


Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)
Linear(in_features=1280, out_features=1000, bias=True)
Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=7, bias=True)
)
Linear(in_features=1280, out_features=7, bias=True)


In [30]:
class_weights = class_weights.to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

print("Device:", device)
print("Number of classes:", NUM_CLASSES)
print(model.classifier)

Device: cuda
Number of classes: 7
Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=7, bias=True)
)


In [31]:
#check if things re working
images, labels = next(iter(train_loader))
images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    logits = model(images)
    loss = criterion(logits, labels)

print("logits:", logits.shape)
print("loss:", loss.item())

logits: torch.Size([32, 7])
loss: 2.0517985820770264


In [32]:
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score
from tqdm.auto import tqdm
import copy


In [33]:
def run_one_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader, leave=False):
        images = images.to(device)
        labels = labels.to(device)

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)

    metrics = {
        "loss": avg_loss,
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro"),
        "balanced_accuracy": balanced_accuracy_score(all_labels, all_preds),
    }

    return metrics

In [34]:
MAX_EPOCHS = 10
PATIENCE = 3

best_val_macro_f1 = -1.0
best_state = None
epochs_without_improvement = 0
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    train_metrics = run_one_epoch(model, train_loader, criterion, optimizer)
    val_metrics = run_one_epoch(model, val_loader, criterion, optimizer=None)

    row = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"train loss {train_metrics['loss']:.4f} | "
        f"train macro-F1 {train_metrics['macro_f1']:.4f} | "
        f"val loss {val_metrics['loss']:.4f} | "
        f"val macro-F1 {val_metrics['macro_f1']:.4f} | "
        f"val balanced acc {val_metrics['balanced_accuracy']:.4f}"
    )

    if val_metrics["macro_f1"] > best_val_macro_f1:
        best_val_macro_f1 = val_metrics["macro_f1"]
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0

        torch.save({
            "epoch": epoch,
            "model_state_dict": best_state,
            "optimizer_state_dict": optimizer.state_dict(),
            "val_macro_f1": best_val_macro_f1,
            "val_metrics": val_metrics,
            "image_size": IMAGE_SIZE,
            "normalization": "imagenet",
            "architecture": "efficientnet_v2_s",
        }, "best_efficientnetv2s_image_only_ham10000.pt")

        print("Saved new best checkpoint.")
    else:
        epochs_without_improvement += 1
        print(f"No improvement: {epochs_without_improvement}/{PATIENCE}")

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping triggered.")
        break

history_df = pd.DataFrame(history)
history_df.to_csv("training_history_efficientnetv2s_image_only.csv", index=False)
history_df

  0%|          | 0/249 [00:00<?, ?it/s]

  0%|          | 0/32 [00:00<?, ?it/s]

Epoch 01 | train loss 1.2183 | train macro-F1 0.3914 | val loss 0.7522 | val macro-F1 0.6263 | val balanced acc 0.7320
Saved new best checkpoint.


  0%|          | 0/249 [00:00<?, ?it/s]

  0%|          | 0/32 [00:00<?, ?it/s]

...

Epoch 08 | train loss 0.2590 | train macro-F1 0.8301 | val loss 0.6141 | val macro-F1 0.7016 | val balanced acc 0.8134
No improvement: 2/3


  0%|          | 0/249 [00:00<?, ?it/s]

  0%|          | 0/32 [00:00<?, ?it/s]

Epoch 09 | train loss 0.2072 | train macro-F1 0.8564 | val loss 0.4915 | val macro-F1 0.7973 | val balanced acc 0.8195
No improvement: 3/3
Early stopping triggered.


,epoch,train_loss,train_accuracy,train_macro_f1,train_balanced_accuracy,val_loss,val_accuracy,val_macro_f1,val_balanced_accuracy
0,1,1.218276,0.549209,0.391374,0.568866,0.752204,0.720884,0.626334,0.731963
1,2,0.665894,0.734873,0.632305,0.763155,0.588732,0.734940,0.667284,0.797969
2,3,0.501214,0.780944,0.706287,0.831157,0.569813,0.781124,0.688110,0.792404
3,4,0.432186,0.808812,0.755516,0.859637,0.545422,0.812249,0.761900,0.808741
4,5,0.375910,0.835174,0.793788,0.885696,0.512639,0.845382,0.802602,0.819059
5,6,0.277875,0.861034,0.840883,0.915665,0.511072,0.878514,0.836985,0.831246
6,7,0.235474,0.865428,0.840452,0.916441,0.433958,0.834337,0.817913,0.873201
7,8,0.259027,0.871454,0.830081,0.912427,0.614128,0.783133,0.701585,0.813433
8,9,0.207210,0.884760,0.856444,0.928223,0.491545,0.849398,0.797298,0.819528


In [36]:
checkpoint = torch.load(
    "best_efficientnetv2s_image_only_ham10000.pt",
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

print("Best epoch:", checkpoint["epoch"])
print("Best val macro-F1:", checkpoint["val_macro_f1"])
print("Best val metrics:", checkpoint["val_metrics"])

Best epoch: 6
Best val macro-F1: 0.836985178255832
Best val metrics: {'loss': 0.5110715023993727, 'accuracy': 0.8785140562248996, 'macro_f1': 0.836985178255832, 'balanced_accuracy': np.float64(0.831245790173398)}


In [37]:

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)

def predict_loader(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in tqdm(loader):
            images = images.to(device)

            logits = model(images)
            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

In [38]:
y_true, y_pred, y_prob = predict_loader(model, test_loader)

test_metrics = {
    "test_accuracy": accuracy_score(y_true, y_pred),
    "test_macro_f1": f1_score(y_true, y_pred, average="macro"),
    "test_weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    "test_balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
}

test_metrics

  0%|          | 0/32 [00:00<?, ?it/s]

{'test_accuracy': 0.8664658634538153,
 'test_macro_f1': 0.8041500178204449,
 'test_weighted_f1': 0.867899919477762,
 'test_balanced_accuracy': np.float64(0.8341980085099351)}

In [39]:
test_metrics_df = pd.DataFrame([test_metrics])
test_metrics_df.to_csv("test_metrics_efficientnetv2s_image_only.csv", index=False)
test_metrics_df

,test_accuracy,test_macro_f1,test_weighted_f1,test_balanced_accuracy
0,0.866466,0.80415,0.8679,0.834198


In [40]:
report_df = pd.DataFrame(classification_report(y_true, y_pred, output_dict=True)).T
report_df.to_csv("test_classification_report_efficientnetv2s_image_only.csv")
report_df

,precision,recall,f1-score,support
0,0.777778,0.848485,0.811594,33.000000
1,0.774194,0.923077,0.842105,52.000000
2,0.792079,0.733945,0.761905,109.000000
3,0.888889,0.727273,0.800000,11.000000
4,0.636364,0.693694,0.663793,111.000000
5,0.939722,0.912913,0.926123,666.000000
6,0.700000,1.000000,0.823529,14.000000
accuracy,0.866466,0.866466,0.866466,0.866466
macro avg,0.787004,0.834198,0.804150,996.000000
weighted avg,0.871817,0.866466,0.867900,996.000000


In [41]:
cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm)
cm_df.to_csv("test_confusion_matrix_efficientnetv2s_image_only.csv")
cm_df

,0,1,2,3,4,5,6
0,28,3,0,1,0,1,0
1,0,48,1,0,2,1,0
2,5,3,80,0,10,10,1
3,0,1,0,8,0,2,0
4,1,0,6,0,77,25,2
5,2,7,14,0,32,608,3
6,0,0,0,0,0,0,14


In [42]:
torch.save(model.state_dict(), "efficientnetv2s_image_only_state_dict.pt")